# 04 - Modelling

LightGBM classifier on the hourly features, evaluated per hour and per patient. `ICULOS`, `patient_id` and `hospital` stay in the tables but are never model inputs.

In [66]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve

from pathlib import Path

PROCESSED_DIR = Path("../data/processed")  # pickles live here (created by the notebooks)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [67]:
test = pd.read_pickle(PROCESSED_DIR / "test.pkl")
train = pd.read_pickle(PROCESSED_DIR / "train.pkl")
validation = pd.read_pickle(PROCESSED_DIR / "validation.pkl")

## Features and target
Inputs: every column except `SepsisLabel`, `patient_id`, `hospital` and `ICULOS`.

In [68]:
x_train = train.drop(["SepsisLabel", "patient_id", "hospital", "ICULOS"], axis=1)
y_train = train["SepsisLabel"]
x_test = test.drop(["SepsisLabel", "patient_id", "hospital", "ICULOS"], axis=1)
y_test = test["SepsisLabel"]
x_val = validation.drop(["SepsisLabel", "patient_id", "hospital", "ICULOS"], axis=1)
y_val = validation["SepsisLabel"]

## Baseline
Large class weight (`scale_pos_weight` about 55). Early stopping watches every metric and the validation log-loss gets worse immediately, so training stops after 1 tree (validation AUROC 0.81, AUPRC 0.076).

In [69]:
params = {
    "n_estimators" : 1000,
    "learning_rate": 0.05,
    "random_state" : 42,
    "scale_pos_weight" : (y_train == 0).sum() / (y_train == 1).sum()
}
model = lgb.LGBMClassifier(**params)
model.fit(x_train, y_train, eval_set=[(x_val, y_val)], eval_metric="average_precision", callbacks=[lgb.early_stopping(50)])

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's average_precision: 0.0757217	valid_0's binary_logloss: 0.0982618


,learning_rate,0.05
,n_estimators,1000
,random_state,42
,scale_pos_weight,np.float64(54.543762781186096)
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [70]:
y_val_ = model.predict_proba(x_val)[:, 1]
print (average_precision_score(y_val, y_val_))
print (roc_auc_score(y_val, y_val_))
print (model.best_iteration_)

0.0757217145836746
0.8095368126118614
1


## Tuning
Early stopping now follows only average precision (AUPRC), with smaller trees (`num_leaves=15`): validation AUROC 0.847, 83 trees.

In [71]:
params.update({
    "num_leaves" : 15,
    "metric": "average_precision"
})
model = lgb.LGBMClassifier(**params)
model.fit(x_train, y_train, eval_set=[(x_val, y_val)], eval_metric="average_precision", callbacks=[lgb.early_stopping(5, first_metric_only=True)])

Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[83]	valid_0's average_precision: 0.140379
Evaluated only: average_precision


,num_leaves,15
,learning_rate,0.05
,n_estimators,1000
,random_state,42
,scale_pos_weight,np.float64(54.543762781186096)
,metric,'average_precision'
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [72]:
y_val_ = model.predict_proba(x_val)[:, 1]
print (roc_auc_score(y_val, y_val_))
print (model.best_iteration_)

0.8470010031946269
83


Lower class weight (5), `min_child_samples=200` and `colsample_bytree=0.5` against overfitting, since hours of the same patient are very similar: validation AUROC 0.849, 45 trees. This model is used from here on.

In [73]:
params.update({
    "scale_pos_weight" : 5,
    "min_child_samples" : 200,
    "colsample_bytree" : 0.5
})
model = lgb.LGBMClassifier(**params)
model.fit(x_train, y_train, eval_set=[(x_val, y_val)], eval_metric="average_precision", callbacks=[lgb.early_stopping(5, first_metric_only=True)])

Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[45]	valid_0's average_precision: 0.143527
Evaluated only: average_precision


,num_leaves,15
,learning_rate,0.05
,n_estimators,1000
,min_child_samples,200
,colsample_bytree,0.5
,random_state,42
,scale_pos_weight,5
,metric,'average_precision'
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000


In [74]:
y_val_ = model.predict_proba(x_val)[:, 1]
print (roc_auc_score(y_val, y_val_))
print (model.best_iteration_)

0.8494618948765232
45


## Choosing a threshold
Precision and recall on validation for thresholds with recall between 0.6 and 0.8, then the F1-optimal threshold (0.33: precision 18%, recall 28% per hour).

In [75]:
precision, recall, thresholds = precision_recall_curve(y_val, y_val_)
precision, recall = precision[:-1], recall[:-1]
mask = (recall >= 0.6) & (recall <= 0.8)
result = pd.DataFrame({
    "threshold": thresholds[mask],
    "precision": precision[mask],
    "recall": recall[mask],
})
result.iloc[::max(len(result) // 10, 1)]

,threshold,precision,recall
0,0.068646,0.051268,0.799952
3344,0.072225,0.053044,0.778255
6688,0.076329,0.055472,0.764902
10032,0.080726,0.058115,0.750119
13376,0.085539,0.060945,0.733429
16720,0.090802,0.064332,0.717454
20064,0.096716,0.067666,0.697186
23408,0.103314,0.071883,0.678827
26752,0.111146,0.076485,0.656414
30096,0.120412,0.080971,0.625179


In [76]:
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = np.argmax(f1)
threshold = thresholds[best]
print(threshold, precision[best], recall[best], f1[best])
print((y_val_ >= threshold).mean())

0.3333568065306092 0.17852308617837268 0.28302336671435385 0.21894309692841984
0.02837790543828051


## Patient-level view
Per-hour metrics understate the usefulness: one alert per patient is enough. At 0.33, 190 of 440 septic validation patients get an alert (median 25.5 h before onset); at 0.1, 387 do, but 1,980 of 5,611 healthy patients get one too. Onset is taken as the first positive label + 6 h.

In [77]:
validation_ = validation[["patient_id", "SepsisLabel"]].copy()
validation_["SepsisLabel_geuss"] = ( y_val_ >= threshold ).astype(int)
validation_.groupby("patient_id").max().groupby(["SepsisLabel", "SepsisLabel_geuss"]).size()

SepsisLabel  SepsisLabel_geuss
0            0                    5486
             1                     125
1            0                     250
             1                     190
dtype: int64

In [78]:
v = validation[["patient_id", "ICULOS", "SepsisLabel"]].copy()
v["alert"] = y_val_ >= threshold
sepsis_ids = v.loc[v["SepsisLabel"] == 1, "patient_id"].unique()
vs = v[v["patient_id"].isin(sepsis_ids)]
first_label = vs[vs["SepsisLabel"] == 1].groupby("patient_id")["ICULOS"].min()
first_alert = vs[vs["alert"]].groupby("patient_id")["ICULOS"].min()
onset = first_label + 6
first_alert = first_alert.reindex(onset.index)
lead = onset - first_alert
print("sepsis patients:", len(onset))
print("any alert:", first_alert.notna().sum())
print("alert before onset:", (lead > 0).sum())
print(lead.describe())

sepsis patients: 440
any alert: 190
alert before onset: 167
count    190.000000
mean      47.810526
std       56.447437
min       -3.000000
25%        6.250000
50%       25.500000
75%       72.750000
max      272.000000
Name: ICULOS, dtype: float64


In [79]:
threshold = 0.1
validation_ = validation[["patient_id", "SepsisLabel"]].copy()
validation_["SepsisLabel_geuss"] = ( y_val_ >= threshold ).astype(int)
validation_.groupby("patient_id").max().groupby(["SepsisLabel", "SepsisLabel_geuss"]).size()

SepsisLabel  SepsisLabel_geuss
0            0                    3631
             1                    1980
1            0                      53
             1                     387
dtype: int64

## Alert rule
An alert is raised only if the probability stays above the threshold for `k` consecutive hours, which suppresses isolated spikes. Grid over `k` in {1, 2, 3} and threshold in {0.1, 0.2, 0.33} on validation.

In [80]:
def alerts_k(df, p, threshold, k):
    a = pd.Series((p >= threshold).astype(int), index=df.index)
    return (a.groupby(df["patient_id"])
              .transform(lambda s: s.rolling(k).min())
              .fillna(0).astype(int))


def evaluate(df, p, threshold, k):
    v = df[["patient_id", "ICULOS", "SepsisLabel"]].copy()
    v["alert"] = alerts_k(df, p, threshold, k)

    sick_ids = v.loc[v["SepsisLabel"] == 1, "patient_id"].unique()
    v["sick"] = v["patient_id"].isin(sick_ids)

    vs = v[v["sick"]]
    onset = vs[vs["SepsisLabel"] == 1].groupby("patient_id")["ICULOS"].min() + 6
    first_alert = (vs[vs["alert"] == 1].groupby("patient_id")["ICULOS"].min()
                     .reindex(onset.index))
    caught = (first_alert < onset).sum()

    healthy = v[~v["sick"]]
    n_healthy = healthy["patient_id"].nunique()
    false_patients = healthy.groupby("patient_id")["alert"].max().sum()
    false_hours = healthy["alert"].sum()

    print(f"k={k}, thr={threshold}")
    print(f"  sepsis: caught before onset {caught}/{len(onset)}, missed {len(onset) - caught}")
    print(f"  healthy: false-alarm patients {false_patients}/{n_healthy}, "
          f"false-alarm hours {false_hours} ({healthy['alert'].mean() * 100:.2f}% of healthy hours)")


for k in (1, 2, 3):
    for thr in (0.1, 0.2, 0.33):
        evaluate(validation, y_val_, thr, k)

k=1, thr=0.1
  sepsis: caught before onset 367/440, missed 73
  healthy: false-alarm patients 1980/5611, false-alarm hours 23642 (11.42% of healthy hours)
k=1, thr=0.2
  sepsis: caught before onset 244/440, missed 196
  healthy: false-alarm patients 574/5611, false-alarm hours 4120 (1.99% of healthy hours)
k=1, thr=0.33
  sepsis: caught before onset 167/440, missed 273
  healthy: false-alarm patients 129/5611, false-alarm hours 767 (0.37% of healthy hours)
k=2, thr=0.1
  sepsis: caught before onset 341/440, missed 99
  healthy: false-alarm patients 1670/5611, false-alarm hours 19564 (9.45% of healthy hours)
k=2, thr=0.2
  sepsis: caught before onset 227/440, missed 213
  healthy: false-alarm patients 424/5611, false-alarm hours 3160 (1.53% of healthy hours)
k=2, thr=0.33
  sepsis: caught before onset 156/440, missed 284
  healthy: false-alarm patients 82/5611, false-alarm hours 577 (0.28% of healthy hours)
k=3, thr=0.1
  sepsis: caught before onset 331/440, missed 109
  healthy: false-

**Chosen: `k=2`, threshold 0.1.** On validation: 341 of 440 septic patients alerted before onset, 9.5% false-alarm hours. A lower threshold catches more sepsis but raises false alarms, so the choice depends on how many false alarms are acceptable. Both values were picked on validation only.

## Test set (evaluated once)

In [81]:
threshold = 0.1
k = 2
p_test = model.predict_proba(x_test)[:, 1]
print("AUPRC:", average_precision_score(y_test, p_test))
print("AUROC:", roc_auc_score(y_test, p_test))
evaluate(test, p_test, threshold, k)

AUPRC: 0.10379484820491212
AUROC: 0.8279924490749687
k=2, thr=0.1
  sepsis: caught before onset 326/440, missed 114
  healthy: false-alarm patients 1686/5610, false-alarm hours 20016 (9.67% of healthy hours)


| Metric | Test |
|---|---|
| AUROC | 0.828 |
| AUPRC | 0.104 (about 1.8% of hours are positive) |
| Septic patients alerted before onset | 326 / 440 (74%) |
| Healthy patients with at least one false alarm | 1,686 / 5,610 (30%) |
| False-alarm hours among healthy patients | 9.7% |

## Feature importance
Number of splits per feature (only 45 trees, so read it as a rough picture). `HospAdmTime` is first; it is constant per patient, so trees may partly use it to recognise individual patients. Clinical variables (`Temp`, `WBC`, `Resp_mean6`, `Creatinine`, `Platelets`) follow.

In [82]:
imp = pd.Series(model.feature_importances_, index=x_train.columns)
imp.sort_values(ascending=False).head(15)

HospAdmTime    46
Temp           28
WBC            18
Unit2          18
Resp_mean6     17
FiO2_since     16
Creatinine     16
HR_count       15
O2Sat_count    15
MAP_count      15
HR             14
SBP_count      14
Platelets      14
SBP_mean6      14
Temp_mean6     12
dtype: int32

## Generalisation to a new hospital
Train on hospital A only (early stopping on A's validation set) and test on hospital B, compared with the main model (trained on A+B) on the same B patients.

In [83]:
drop_cols = ["SepsisLabel", "patient_id", "hospital", "ICULOS"]

train_A = train[train["hospital"] == "A"]
val_A = validation[validation["hospital"] == "A"]
test_B = test[test["hospital"] == "B"]

model_A = lgb.LGBMClassifier(**params)
model_A.fit(
    train_A.drop(columns=drop_cols), train_A["SepsisLabel"],
    eval_set=[(val_A.drop(columns=drop_cols), val_A["SepsisLabel"])],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50, first_metric_only=True)],
)

x_B = test_B.drop(columns=drop_cols)
y_B = test_B["SepsisLabel"]

p_B_from_A = model_A.predict_proba(x_B)[:, 1]
p_B_full = p_test[(test["hospital"] == "B").values]

for name, p_ in [("trained on A only", p_B_from_A), ("trained on A+B", p_B_full)]:
    print(name)
    print("  AUPRC:", average_precision_score(y_B, p_))
    print("  AUROC:", roc_auc_score(y_B, p_))
    evaluate(test_B, p_, 0.1, 2)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[138]	valid_0's average_precision: 0.149038
Evaluated only: average_precision
trained on A only
  AUPRC: 0.05685416919034943
  AUROC: 0.7486966298015457
k=2, thr=0.1
  sepsis: caught before onset 86/166, missed 80
  healthy: false-alarm patients 719/2797, false-alarm hours 6753 (6.55% of healthy hours)
trained on A+B
  AUPRC: 0.10410466402822766
  AUROC: 0.8400007872674756
k=2, thr=0.1
  sepsis: caught before onset 113/166, missed 53
  healthy: false-alarm patients 598/2797, false-alarm hours 6737 (6.53% of healthy hours)


| Test on hospital B | AUROC | AUPRC | Septic alerted before onset | Healthy with false alarm |
|---|---|---|---|---|
| Trained on A only | 0.749 | 0.057 | 86 / 166 | 719 / 2,797 |
| Trained on A+B | 0.840 | 0.104 | 113 / 166 | 598 / 2,797 |

The A-only model loses a lot on the unseen hospital (AUROC 0.84 to 0.75, AUPRC 0.104 to 0.057), so hospital-specific recording practice matters. Part of the gap is simply that B was absent from training.

## Limitations
- "Alerted before onset" counts any earlier alert, even days earlier; no lead-time window is applied for the final rule.
- Threshold and `k` are a trade-off, not tuned for a clinical target.
- One split, no confidence intervals, no probability calibration.
- Row-level precision is low: 74% coverage costs many false alarms.